In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from dllm.core.schedulers import LinearAlphaScheduler
from dllm.core.samplers.mdlm import MDLMSampler, MDLMSamplerConfig


/home/prasanna/coding/diff-decoder/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'lm_eval'

In [ ]:
# 1. Initialize your custom/HuggingFace model and Tokenizer
# (Assuming the model is a Masked Diffusion LM like LLaDA)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForMaskedLM.from_pretrained("dllm-hub/Qwen3-0.6B-diffusion-bd3lm-v0.1", dtype=torch.bfloat16, trust_remote_code=True).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("dllm-hub/Qwen3-0.6B-diffusion-bd3lm-v0.1", trust_remote_code=True)

# Mock scheduler (often instantiated within your framework; MDLMSampler uses self.scheduler)
scheduler = LinearAlphaScheduler()

# 2. Configure the Sampler
config = MDLMSamplerConfig(
    max_new_tokens=64, 
    steps=64,              # Take 64 steps to fully unmask the generated text
    block_size=64,         # Process in blocks of 64
    temperature=0.1,       # Slight stochasticity
    cfg_scale=1.5,         # Classifier-free guidance multiplier
    remasking="low_confidence",
    return_dict=True       # Returns BaseSamplerOutput instead of pure tensor
)

In [ ]:
# 3. Instantiate the MDLMSampler
sampler = MDLMSampler(
    model=model, 
    tokenizer=tokenizer, 
    scheduler=scheduler
)

# ---------------------------------------------------------
# Scenario A: Standard Text Generation (Appending)
# ---------------------------------------------------------
prompts = ["The capital of France is"]
tokenized_inputs = [tokenizer.encode(p) for p in prompts]

output = sampler.sample(
    inputs=tokenized_inputs,
    config=config
)

print("Generation Result:")
print(tokenizer.decode(output.sequences[0]))

# ---------------------------------------------------------
# Scenario B: Text Infilling (Replacing masks in context)
# ---------------------------------------------------------
mask_id = tokenizer.mask_token_id
# Creating an input: "I went to the [MASK] [MASK] to buy some groceries."
infill_tokens = tokenizer.encode("I went to the ") + [mask_id, mask_id] + tokenizer.encode(" to buy some groceries.", add_special_tokens=False)

infill_output = sampler.infill(
    inputs=[infill_tokens],
    config=config
)

print("\nInfill Result:")
print(tokenizer.decode(infill_output.sequences[0]))

In [1]:
import torch
import torch.nn as nn

def mask_tokens(tokens, mask_token_id, p=0.3):
    masked = tokens.clone()
    mask = torch.rand(tokens.shape) < p
    masked[mask] = mask_token_id
    return masked, mask

def diffusion_loss(model, tokens, mask_token_id):
    xt, mask = mask_tokens(tokens, mask_token_id)
    logits = model(xt).logits
    loss = torch.nn.functional.cross_entropy(
        logits[mask],
        tokens[mask]
    )
    return loss

# 1. Create a simple mock model that behaves like a Hugging Face model
class MockHFModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Simulates going from tokens -> hidden states -> logits
        self.embedding = nn.Embedding(vocab_size, 32)
        self.lm_head = nn.Linear(32, vocab_size)

    def forward(self, input_ids):
        hidden_states = self.embedding(input_ids)
        logits = self.lm_head(hidden_states) # Shape: (batch_size, seq_len, vocab_size)
        
        # Simulating Hugging Face model output format (returns an object with .logits)
        class ModelOutput:
            pass
        out = ModelOutput()
        out.logits = logits
        
        return out

In [1]:
# Fix seed for reproducibility in results
torch.manual_seed(42)

vocab_size = 100
mask_token_id = 99
batch_size = 2
seq_len = 10

# Create random token sequences
tokens = torch.randint(0, mask_token_id, (batch_size, seq_len))
print("=== Original Tokens ===")
print(tokens)

# Initialize the mock model
model = MockHFModel(vocab_size)

# --- Test mask_tokens ---
print("\n=== Testing mask_tokens ===")
masked_tokens, mask = mask_tokens(tokens, mask_token_id, p=0.5)

print("Masked Tokens (99 is the mask):")
print(masked_tokens)
print("\nBoolean Mask:")
print(mask)

# --- Test diffusion_loss ---
print("\n=== Testing diffusion_loss ===")

# Calculate loss
loss = diffusion_loss(model, tokens, mask_token_id)
print(f"Calculated Loss: {loss.item():.4f}")

# The shape formatting inside diffusion loss works because cross_entropy expects:
# input: (N, C) where N is the number of masked items and C is vocab_size
# target: (N) where N is the number of masked ground truth labels
# By indexing `logits[mask]`, PyTorch flattens the batch and seq dims for masked items only.

NameError: name 'torch' is not defined